# Radar through the winter — Baltic sea ice with Sentinel-1

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwinkelmann/GEE_Animation/blob/main/notebooks/09_sar_baltic_ice.ipynb)

Every animation so far used **optical** sensors, and all of them share one
weakness: clouds. A Baltic winter is their worst case — weeks of overcast and
eight hours of grey daylight. **Synthetic-aperture radar (SAR)** does not care:
it brings its own illumination and its microwaves pass through clouds, so
Sentinel-1 images the sea ice in the Bodden lagoons on every single pass, day
and night, December through April.

This notebook animates the **winter 2025/26** over the Rügen–Usedom coast. The
radar never joined the package's sensor registry (its physics fits none of the
optical conventions — no clouds to mask, backscatter instead of reflectance),
so the Earth Engine work here is raw and explicit; from the package we take
auth, AOI handling and the frame annotation.

## 1 · Install (Colab only)

In [ ]:
import sys
IS_COLAB = "google.colab" in sys.modules
BRANCH = "main"
URL = f"gee_animation[shapefile] @ git+https://github.com/cwinkelmann/GEE_Animation.git@{BRANCH}"
if IS_COLAB:
    !pip install -q "{URL}"                              # deps + package (first run)
    !pip install -q --force-reinstall --no-deps "{URL}"  # branch moved? force-refresh our code

## 2 · Authenticate

In [ ]:
import os
PROJECT = os.environ.get("EE_PROJECT", "")   # <-- or paste your own project id here
assert PROJECT, "Set PROJECT to your own Earth Engine Cloud project id — 00_setup.ipynb shows how to create one"

from gee_animation import auth
auth.init(PROJECT)

## 3 · The scene and the season

The AOI spans the shallow lagoons between Rügen and Usedom — the Greifswalder
Bodden freezes in most cold spells while the open Baltic beside it stays liquid,
which is exactly the contrast radar shows well.

In [ ]:
from pathlib import Path
import ee
from gee_animation import aoi

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs" / "aoi").is_dir()), None)
if REPO is not None:
    REGION = str(REPO / "docs" / "aoi" / "balticsea_ice" / "aoi.geojson")
else:   # Colab: fetch the same file
    import urllib.request
    RAW = (f"https://raw.githubusercontent.com/cwinkelmann/GEE_Animation/{BRANCH}"
           "/docs/aoi/balticsea_ice/aoi.geojson")
    urllib.request.urlretrieve(RAW, "aoi.geojson")
    REGION = "aoi.geojson"

START, END = "2025-12-01", "2026-05-01"      # the ice season, end exclusive

region_geom = aoi.parse({"geojson": REGION})
bbox = region_geom.bounds().coordinates().get(0).getInfo()
print("AOI corners:", [[round(c, 2) for c in p] for p in bbox[:-1]])

## 4 · Ask the archive before assuming anything

Sentinel-1 scenes differ by **instrument mode** (IW over most land and coasts,
EW in the Arctic) and **polarisation** (VV, VH, HH …). Which combination
actually covers *this* AOI *this* winter is a question for the archive, not for
a tutorial's assumption:

In [ ]:
s1 = (ee.ImageCollection("COPERNICUS/S1_GRD")
      .filterBounds(region_geom)
      .filterDate(START, END))

combos = s1.aggregate_histogram(
    "instrumentMode").getInfo()
pols = s1.aggregate_histogram("transmitterReceiverPolarisation").getInfo()
print("scenes by mode:        ", combos)
print("scenes by polarisation:", pols)

BAND = "VV"          # chosen from the inventory above: IW VV+VH dominates here
s1 = (s1.filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", BAND))
        .select(BAND))
print(f"usable {BAND} scenes:", s1.size().getInfo())

## 5 · Ten-day backscatter mosaics

Sea ice moves and grows on a scale of days, so monthly frames would blur the
story. Ten-day medians keep the motion and still smooth the speckle a little.
`S1_GRD` delivers backscatter in **dB** (a log scale); the median of dB values
is fine for a picture — just remember it is not the mean of the physical power.

Reading the frames: wind-roughened **open water scatters strongly** (bright);
**smooth new ice is dark**; ridged or snow-covered ice sits in between. Land is
bright year-round.

In [ ]:
import io
import urllib.request
import numpy as np
from datetime import date, timedelta
from PIL import Image as PILImage
from gee_animation.render import annotate

VIZ = {"min": -25, "max": 0}                 # dB, fixed across all frames
STEP = 10                                     # days per frame

def periods(start, end, step):
    d0, d1 = date.fromisoformat(start), date.fromisoformat(end)
    while d0 < d1:
        dn = min(d0 + timedelta(days=step), d1)
        if (dn - d0).days >= step // 2:      # drop a trailing stub period
            yield str(d0), str(dn)
        d0 = dn

frames, labels = [], []
for p0, p1 in periods(START, END, STEP):
    chunk = s1.filterDate(p0, p1)
    n = chunk.size().getInfo()
    if n == 0:
        print(f"{p0} … skipped (no scenes)")
        continue
    mosaic = chunk.median().clip(region_geom)
    url = mosaic.getThumbURL({**VIZ, "region": region_geom, "dimensions": 900,
                              "format": "png"})
    png = PILImage.open(io.BytesIO(urllib.request.urlopen(url).read())).convert("RGBA")
    canvas = PILImage.new("RGBA", png.size, (40, 40, 40, 255))   # no-data -> dark grey
    canvas.alpha_composite(png)
    rgb = np.array(canvas.convert("RGB"))
    frames.append(annotate(rgb, f"{p0} · {n} passes", credit="Sentinel-1 VV, contains modified Copernicus data"))
    print(f"{p0} … {n} scenes")
print(len(frames), "frames")

## 6 · Assemble and watch

In [ ]:
import imageio.v2 as imageio

# Crop to multiples of 16 so the H.264 encoder needs no resampling.
h16, w16 = (frames[0].shape[0] // 16) * 16, (frames[0].shape[1] // 16) * 16
frames = [f[:h16, :w16] for f in frames]

out = Path("out"); out.mkdir(exist_ok=True)
gif = out / "baltic_ice_2025_26.gif"
imageio.mimwrite(gif, frames, duration=1000 / 2, loop=0)        # 2 frames/s
try:
    mp4 = out / "baltic_ice_2025_26.mp4"
    imageio.mimwrite(mp4, frames, fps=2, codec="libx264", quality=8)
    print("wrote", gif, "and", mp4)
except Exception as e:                       # ffmpeg missing -> GIF still stands
    mp4 = None
    print("wrote", gif, f"(mp4 skipped: {e})")

from IPython.display import Image as IPyImage, display
display(IPyImage(filename=str(gif)))

## 7 · What optical would have given you

The same AOI, the same winter, Sentinel-2. Counting granules under the series'
usual 70 % cloud filter gives flattering numbers: the AOI touches five 100 km
tiles seen from three orbits, so one clear day yields five or more granules, and a
granule that only clips the AOI's edge counts the same as a full view. So count
**days**, and only days whose granules together cover most of the AOI. Footprints
and the cloud percentage are metadata — one request fetches them all, the
geometry is done locally — and the case for radar makes itself:

Then the same animation again from optical: per 10-day period the least cloudy
full pass, true colour, next to the radar frame of that period.

In [ ]:
import json
from collections import defaultdict
from shapely.geometry import shape
from shapely.ops import unary_union, transform
from pyproj import Transformer

MIN_COVER = 0.7                                     # a pass must see this share of the AOI
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(region_geom).filterDate(START, END))
granules = s2.map(lambda im: ee.Feature(im.geometry(), {
    "date": im.date().format("YYYY-MM-dd"),
    "cloud": im.get("CLOUDY_PIXEL_PERCENTAGE")})).getInfo()["features"]   # one call, metadata only

to_m = Transformer.from_crs("EPSG:4326", "EPSG:32633", always_xy=True).transform   # UTM 33N, metres
aoi_m = transform(to_m, shape(json.load(open(REGION))["features"][0]["geometry"]))
by_day = defaultdict(list)
for f in granules:
    part = transform(to_m, shape(f["geometry"])).intersection(aoi_m)
    if not part.is_empty:
        by_day[f["properties"]["date"]].append((part, f["properties"]["cloud"]))

days = []                                           # (date, AOI share covered, area-weighted granule cloud %)
for d, parts in sorted(by_day.items()):
    cover = unary_union([p for p, _ in parts]).area / aoi_m.area
    cloud = sum(p.area * c for p, c in parts) / sum(p.area for p, _ in parts)
    if cover >= MIN_COVER:
        days.append((d, cover, cloud))
print(f"{len(by_day)} days with any granule, {len(days)} of them cover >= {MIN_COVER:.0%} of the AOI")
for m in sorted({d[:7] for d in by_day}):
    full = [x for x in days if x[0].startswith(m)]
    print(f"{m}: {len(full):2d} full passes, {sum(c <= 30 for _, _, c in full)} with <= 30 % cloud")
print("radar passes in the same window:", s1.size().getInfo())

In [ ]:
# Per period (same STEP as the radar frames) the least cloudy full pass, true colour.
# A period without a full pass gets an empty frame with a hollow marker.
RGB = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}
optical = []
for p0, p1 in periods(START, END, STEP):
    passes = [x for x in days if p0 <= x[0] < p1]
    if not passes:
        rgb = np.full((h16, w16, 3), 40, dtype=np.uint8)
        label, real = f"{p0} · no full optical pass", False
    else:
        d, cover, cloud = min(passes, key=lambda x: x[2])
        mosaic = s2.filterDate(d, ee.Date(d).advance(1, "day")).mosaic().clip(region_geom)
        url = mosaic.getThumbURL({**RGB, "region": region_geom, "dimensions": 900, "format": "png"})
        png = PILImage.open(io.BytesIO(urllib.request.urlopen(url).read())).convert("RGBA")
        canvas = PILImage.new("RGBA", png.size, (40, 40, 40, 255))
        canvas.alpha_composite(png)
        rgb = np.array(canvas.convert("RGB"))[:h16, :w16]
        label, real = f"{p0} · best pass {d}, {cloud:.0f} % cloud", True
    optical.append(annotate(rgb, label, credit="Sentinel-2 true colour, contains modified Copernicus data", is_real=real))
    print(label)

pairs = [np.concatenate([r, o], axis=1) for r, o in zip(frames, optical)]   # radar left, optical right
gif2 = out / "baltic_ice_radar_vs_optical_2025_26.gif"
imageio.mimwrite(gif2, pairs, duration=1000 / 2, loop=0)
imageio.mimwrite(out / "baltic_ice_optical_2025_26.gif", optical, duration=1000 / 2, loop=0)
try:
    imageio.mimwrite(out / "baltic_ice_radar_vs_optical_2025_26.mp4", pairs, fps=2, codec="libx264", quality=8)
except Exception as e:
    print(f"mp4 skipped: {e}")
print("wrote", gif2)
display(IPyImage(filename=str(gif2)))

## 8 · Optional: the numbers as GeoTIFF

The frames above are pictures: 8-bit, stretched to the −25…0 dB ramp, no
georeferencing. For QGIS, a classification or a threshold of your own you want
the values. Flip `EXPORT_GEOTIFF` to `True` and re-run the cell: one float32
GeoTIFF per period, VV backscatter in dB, masked pixels as −9999, on a metric
grid in UTM 33N. `GEOTIFF_SCALE` is 40 m (the median of a 10-day stack has no
10 m detail worth the bytes); the cell coarsens it if the file would exceed Earth
Engine's 48 MiB synchronous download cap. Files land in `out/geotiffs/baltic_ice/`.

In [ ]:
EXPORT_GEOTIFF = False                      # set True and re-run this cell
GEOTIFF_SCALE, CRS = 40, "EPSG:32633"       # metres; UTM 33N covers the AOI
NODATA = -9999

if EXPORT_GEOTIFF:
    import math
    from gee_animation.render import EE_SYNC_DOWNLOAD_MAX_BYTES
    tif_dir = Path("out") / "geotiffs" / "baltic_ice"
    tif_dir.mkdir(parents=True, exist_ok=True)

    lons, lats = [p[0] for p in bbox], [p[1] for p in bbox]       # AOI bounds from section 3
    w_m = (max(lons) - min(lons)) * 111320 * math.cos(math.radians(sum(lats) / len(lats)))
    h_m = (max(lats) - min(lats)) * 111320
    scale = GEOTIFF_SCALE
    predicted = (w_m / scale) * (h_m / scale) * 4                  # float32, one band
    if predicted > EE_SYNC_DOWNLOAD_MAX_BYTES:                     # EE refuses, does not retry
        scale = round(scale * math.sqrt(predicted / EE_SYNC_DOWNLOAD_MAX_BYTES), 1)
        print(f"{GEOTIFF_SCALE} m would be {predicted / 2**20:.0f} MiB per file; exporting at {scale} m")

    for p0, p1 in periods(START, END, STEP):
        chunk = s1.filterDate(p0, p1)
        if chunk.size().getInfo() == 0:
            continue
        image = chunk.median().clip(region_geom).unmask(NODATA).toFloat()
        url = image.getDownloadURL({"region": region_geom, "scale": scale, "crs": CRS,
                                    "format": "GEO_TIFF", "filePerBand": False})
        path = tif_dir / f"baltic_ice_{BAND}_{p0}.tif"
        path.write_bytes(urllib.request.urlopen(url).read())
        arr = np.array(PILImage.open(path), dtype="float32")        # PIL reads a float32 TIFF
        valid = arr[arr != NODATA]
        print(f"{path.name}: {arr.shape[1]}x{arr.shape[0]} px, {path.stat().st_size / 2**20:.1f} MiB, "
              f"VV {valid.min():.1f} to {valid.max():.1f} dB, {valid.size / arr.size:.0%} valid")
else:
    print("EXPORT_GEOTIFF is False — nothing written; set it to True for the files")

## Make it yours

- **VH instead of VV** (`BAND = "VH"`, viz max ≈ −5): cross-polarisation darkens
  open water further and makes deformed ice stand out.
- **A real pack-ice winter**: move the AOI to the Bothnian Bay (around
  `[20.5, 64.5, 25.5, 65.9]`) where the Baltic freezes every year, shift the viz,
  and re-run — the archive inventory cell will tell you if EW mode takes over.
- **Another season**: the 2024/25 or 2026/27 winter is one date change away.
- **Sharper time**: `STEP = 6` matches Sentinel-1's repeat cycle; frames get
  noisier but the ice edge moves almost scene by scene.
- Compare your animation to the
  [FMI ice charts](https://en.ilmatieteenlaitos.fi/ice-conditions) for the same
  weeks — the professional ground truth for this sea.